# PowerOps_v1 — Notebook 06: Confidence and Human Escalation

Notebook 05 exposed a real gap: for every one of its 7 "insufficient
evidence" test questions, `retrieve_powerops_documents()` still returned 5
documents (the nearest semantic matches — just not actually relevant ones),
so a placeholder rule of *"escalate only when zero documents were
retrieved"* never fired, even though the LLM's own answer correctly said it
couldn't find enough information. `needs_escalation` was `False` in every
one of those cases — wrong.

This notebook fixes that with **deterministic, evidence-based checks** that
don't depend on the LLM accurately self-reporting confidence:

```text
User Question
      |
   Retrieve
      |
 Enough evidence?
   /        \
 YES         NO
  |           |
Answer   Create escalation
              |
      Management Review
```

### Why not just trust the LLM's confidence?

Because the LLM's stated confidence and its actual grounding can diverge in
both directions — it might say "not found" when better filtering would have
found the answer, or it might state something confidently that isn't
actually supported by the retrieved text. `should_escalate()` below checks
concrete, checkable facts about the retrieval and the answer text itself
(does the requested issue key actually appear in what came back? does every
Issue Key the answer cites actually appear in the retrieved context? is the
requested team/assignee/status combination actually reflected in any
retrieved record?) — not "how confident did the model sound."


## 1. Configuration, vocabulary, and Pinecone/LLM connection

In [1]:
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

load_dotenv(dotenv_path=Path("../.env"))

PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
PINECONE_INDEX_NAME = os.environ.get("PINECONE_INDEX_NAME", "powerops-v1")
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "text-embedding-3-small")
CHAT_MODEL = os.environ.get("CHAT_MODEL", "gpt-4o-mini")
LLM_TEMPERATURE = float(os.environ.get("LLM_TEMPERATURE", "0.0"))
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
TOP_K = int(os.environ.get("TOP_K", "5"))

with open("../data/vocabulary.json", "r", encoding="utf-8") as f:
    vocabulary = json.load(f)

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, api_key=OPENAI_API_KEY)
vector_store = PineconeVectorStore(index=index, embedding=embeddings)
llm = ChatOpenAI(model=CHAT_MODEL, temperature=LLM_TEMPERATURE)

print(f"Connected. Chat model: {CHAT_MODEL}")


Connected. Chat model: gpt-4o-mini


## 2. Retrieval + RAG functions (from Notebooks 04–05)

In [2]:
ISSUE_KEY_PATTERN = re.compile(r"\bINO-\d+\b", re.IGNORECASE)


def find_issue_key(question: str) -> str | None:
    match = ISSUE_KEY_PATTERN.search(question)
    return match.group(0).upper() if match else None


def find_vocab_match(question: str, values: list[str]) -> str | None:
    q_lower = question.lower()
    for value in sorted(values, key=len, reverse=True):
        pattern = r"\b" + re.escape(value.lower()) + r"\b"
        if re.search(pattern, q_lower):
            return value
    return None


def _split_name(assignee: str) -> tuple[str, str]:
    cleaned = assignee.replace("(Contractor)", "").strip()
    if "," in cleaned:
        last, first = (p.strip() for p in cleaned.split(",", 1))
    else:
        parts = cleaned.split()
        last, first = (parts[0], " ".join(parts[1:])) if parts else ("", "")
    return last, first


NAME_PARTS = {a: _split_name(a) for a in vocabulary["assignees"]}


def find_assignee(question: str) -> dict:
    q_lower = question.lower()
    full_matches = set()
    for assignee, (last, first) in NAME_PARTS.items():
        if not last or not first:
            continue
        if (re.search(r"\b" + re.escape(last.lower()) + r"\b", q_lower)
                and re.search(r"\b" + re.escape(first.lower()) + r"\b", q_lower)):
            full_matches.add(assignee)
    if len(full_matches) == 1:
        return {"assignee": next(iter(full_matches)), "ambiguous_candidates": None}
    if len(full_matches) > 1:
        return {"assignee": None, "ambiguous_candidates": sorted(full_matches)}

    token_matches = set()
    for assignee, (last, first) in NAME_PARTS.items():
        for token in (first, last):
            if token and len(token) >= 3 and re.search(r"\b" + re.escape(token.lower()) + r"\b", q_lower):
                token_matches.add(assignee)
                break
    if len(token_matches) == 1:
        return {"assignee": next(iter(token_matches)), "ambiguous_candidates": None}
    if len(token_matches) > 1:
        return {"assignee": None, "ambiguous_candidates": sorted(token_matches)}
    return {"assignee": None, "ambiguous_candidates": None}


def parse_query_filters(question: str) -> dict:
    filters = {}
    matched_spans = []

    issue_key = find_issue_key(question)
    if issue_key:
        filters["issue_key"] = issue_key
        matched_spans.append(issue_key)

    team = find_vocab_match(question, vocabulary["assigned_teams"])
    if team:
        filters["assigned_team"] = team
        matched_spans.append(team)

    priority = find_vocab_match(question, vocabulary["priorities"])
    if priority:
        filters["priority"] = priority
        matched_spans.append(priority)

    status = find_vocab_match(question, vocabulary["statuses"])
    if status:
        filters["status"] = status
        matched_spans.append(status)

    assignee_result = find_assignee(question)
    if assignee_result["assignee"]:
        filters["assignee"] = assignee_result["assignee"]
        last, first = NAME_PARTS[assignee_result["assignee"]]
        matched_spans.extend([t for t in (last, first) if t])

    semantic_query = question
    for span in matched_spans:
        semantic_query = re.sub(re.escape(span), "", semantic_query, flags=re.IGNORECASE)
    semantic_query = re.sub(r"\s+", " ", semantic_query).strip(" ?.")
    if not semantic_query:
        semantic_query = question

    return {
        "filters": filters,
        "ambiguous_assignee_candidates": assignee_result["ambiguous_candidates"],
        "semantic_query": semantic_query,
    }


def build_pinecone_filter(filters: dict) -> dict:
    pinecone_filter = {}
    for key in ("assigned_team", "assignee", "priority", "status", "issue_key"):
        if key in filters:
            pinecone_filter[key] = {"$eq": filters[key]}
    return pinecone_filter


def retrieve_powerops_documents(question: str, top_k: int = TOP_K) -> dict:
    parsed = parse_query_filters(question)
    pinecone_filter = build_pinecone_filter(parsed["filters"])
    results = vector_store.similarity_search_with_score(
        parsed["semantic_query"], k=top_k, filter=pinecone_filter or None,
    )
    return {
        "question": question,
        "filters": parsed["filters"],
        "ambiguous_assignee_candidates": parsed["ambiguous_assignee_candidates"],
        "semantic_query": parsed["semantic_query"],
        "pinecone_filter": pinecone_filter,
        "results": results,
    }


SYSTEM_PROMPT = """You are PowerOps, a DevOps management assistant.

Answer the user's question using ONLY the supplied PowerOps context below.

Do not invent issues, statuses, assignees, priorities, teams, dates, or \
resolutions that are not explicitly present in the context.

When referencing an issue, include its Issue Key (format: INO-#####).

If the available context does not contain enough evidence to answer the \
question, explicitly state that sufficient information was not found in \
the PowerOps knowledge base. Do not guess or fill gaps.

For questions requesting multiple issues, provide a concise table when \
appropriate (columns: Issue Key, Summary, Team, Assignee, Priority, Status)."""


def build_context(docs: list) -> str:
    blocks = [f"[Result {i + 1}]\n{doc.page_content}" for i, doc in enumerate(docs)]
    return "\n\n---\n\n".join(blocks)


print("Retrieval + RAG functions loaded.")


Retrieval + RAG functions loaded.


## 3. `evaluate_evidence()` — deterministic evidence checks

Six concrete, checkable signals — none of them a self-reported confidence
score:

1. **Zero results** — nothing was retrieved at all.
2. **Requested issue key not found** — the question named a specific
   `Issue Key`, but that exact key isn't among the retrieved documents
   (meaning it doesn't exist in this dataset — retrieval fell back to
   semantically-nearest-but-wrong documents instead).
3. **Requested filter combination not reflected** — the question specified a
   team/assignee/priority/status, but *no* retrieved document actually
   satisfies that exact combination (a defensive double-check on top of the
   Pinecone filter itself).
4. **Ambiguous assignee** — a name in the question matches more than one
   real person, and we deliberately didn't guess which one.
5. **LLM indicated insufficient context** — the answer text itself contains
   one of the phrases our system prompt instructs it to use when it can't
   answer (`"sufficient information was not found"`, `"not enough
   information"`, etc.) — checked with plain string matching, not a second
   LLM call asking "are you sure?"
6. **Unsupported citations** — the answer cites an `Issue Key` that isn't
   actually present in the retrieved context (a real hallucination signal).

**A deliberate omission:** raw cosine similarity score thresholding. Notebook
03/04 showed that even *good* semantic matches on this dataset only reach
~0.35–0.55 (short, terse Summaries compress the similarity range), so a fixed
similarity cutoff would either miss real problems or flag good matches as
weak. We still report `max_similarity` for visibility, but it isn't used as
an escalation trigger on its own.


In [3]:
LLM_DECLINE_PHRASES = [
    "sufficient information was not found",
    "not enough information",
    "insufficient information",
    "could not find",
    "cannot determine",
    "can't determine",
    "no information",
    "not found in the powerops knowledge base",
    "unable to determine",
]


def _llm_declined(answer: str) -> bool:
    answer_lower = answer.lower()
    return any(phrase in answer_lower for phrase in LLM_DECLINE_PHRASES)


def _unsupported_citations(answer: str, retrieved_issue_keys: set[str]) -> list[str]:
    cited = {m.upper() for m in ISSUE_KEY_PATTERN.findall(answer)}
    return sorted(cited - retrieved_issue_keys)


def evaluate_evidence(
    question: str,
    retrieved: list[tuple],
    filters: dict,
    ambiguous_assignee_candidates: list | None,
    answer: str,
) -> dict:
    """Compute deterministic evidence signals for one PowerOps answer."""
    docs = [doc for doc, _score in retrieved]
    scores = [score for _doc, score in retrieved]
    retrieved_issue_keys = {doc.metadata["issue_key"] for doc in docs}

    requested_issue_key_not_found = False
    if filters.get("issue_key"):
        requested_issue_key_not_found = filters["issue_key"] not in retrieved_issue_keys

    structured_keys = [k for k in ("assigned_team", "assignee", "priority", "status") if k in filters]
    requested_filters_not_reflected = False
    if structured_keys and docs:
        satisfied = any(
            all(doc.metadata.get(k) == filters[k] for k in structured_keys) for doc in docs
        )
        requested_filters_not_reflected = not satisfied

    return {
        "num_retrieved": len(docs),
        "zero_results": len(docs) == 0,
        "max_similarity": max(scores) if scores else 0.0,
        "requested_issue_key_not_found": requested_issue_key_not_found,
        "requested_filters_not_reflected": requested_filters_not_reflected,
        "ambiguous_assignee": bool(ambiguous_assignee_candidates),
        "llm_declined": _llm_declined(answer),
        "unsupported_citations": _unsupported_citations(answer, retrieved_issue_keys),
    }


## 4. `should_escalate()` — turn evidence signals into an escalation decision

In [4]:
def should_escalate(
    question: str,
    retrieved: list[tuple],
    filters: dict,
    ambiguous_assignee_candidates: list | None,
    answer: str,
) -> tuple[bool, list[str], dict]:
    """Decide whether a PowerOps answer should be escalated to human review.

    Returns (should_escalate, reasons, evidence).
    """
    evidence = evaluate_evidence(question, retrieved, filters, ambiguous_assignee_candidates, answer)
    reasons = []

    if evidence["zero_results"]:
        reasons.append("No relevant PowerOps records were retrieved for this question.")
    if evidence["requested_issue_key_not_found"]:
        reasons.append(f"Requested issue key '{filters.get('issue_key')}' was not found in the PowerOps knowledge base.")
    if evidence["requested_filters_not_reflected"]:
        reasons.append("No retrieved records match the requested team/assignee/priority/status combination.")
    if evidence["ambiguous_assignee"]:
        reasons.append(f"The assignee reference is ambiguous and matches multiple people: {ambiguous_assignee_candidates}.")
    if evidence["llm_declined"]:
        reasons.append("The language model indicated the retrieved context was insufficient to answer confidently.")
    if evidence["unsupported_citations"]:
        reasons.append(f"The generated answer cited Issue Key(s) not present in the retrieved evidence: {evidence['unsupported_citations']}.")

    return (len(reasons) > 0, reasons, evidence)


## 5. `create_escalation()` — persist to `data/escalations.json`

Escalations accumulate in a flat JSON array. Each record captures everything
a human reviewer (or, later, a Jira/ServiceNow/Teams integration) would need
to act on the request without re-running PowerOps.


In [5]:
ESCALATIONS_PATH = Path("../data/escalations.json")


def _load_escalations() -> list[dict]:
    if not ESCALATIONS_PATH.exists():
        return []
    with open(ESCALATIONS_PATH, "r", encoding="utf-8") as f:
        return json.load(f)


def _save_escalations(escalations: list[dict]) -> None:
    with open(ESCALATIONS_PATH, "w", encoding="utf-8") as f:
        json.dump(escalations, f, indent=2)


def create_escalation(
    question: str,
    reasons: list[str],
    evidence: dict,
    retrieved_docs: list | None = None,
    user: str | None = None,
) -> dict:
    """Create and persist a new escalation record. Returns the created record."""
    escalations = _load_escalations()
    escalation_id = f"ESC-{len(escalations) + 1:04d}"
    retrieved_issue_ids = sorted({doc.metadata["issue_key"] for doc in (retrieved_docs or [])})

    record = {
        "escalation_id": escalation_id,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "question": question,
        "user": user,
        "reason": "; ".join(reasons),
        "reasons": reasons,
        "retrieved_issue_ids": retrieved_issue_ids,
        "evidence": evidence,
        "status": "Pending Management Review",
    }

    escalations.append(record)
    _save_escalations(escalations)
    return record


print("create_escalation() defined. Escalations will be saved to", ESCALATIONS_PATH.resolve())


create_escalation() defined. Escalations will be saved to /Users/svedamur/Documents/agentic-ai-simulations/devOps_rag_week2/data/escalations.json


## 6. `ask_powerops()` v2 — now with real escalation logic

This supersedes Notebook 05's version. The zero-document short-circuit is
still there (cheapest, most certain case), but now every answer — including
ones the LLM actually generated — is checked by `should_escalate()` before
being returned.


In [6]:
def ask_powerops(question: str, top_k: int = TOP_K, user: str | None = None) -> dict:
    """Answer a PowerOps question, grounded in retrieved evidence, with escalation.

    Returns:
        {
            "question": str,
            "answer": str,
            "sources": list[str],
            "filters": dict,
            "retrieved_count": int,
            "needs_escalation": bool,
            "escalation": dict | None,
        }
    """
    retrieval = retrieve_powerops_documents(question, top_k=top_k)
    retrieved = retrieval["results"]
    docs = [doc for doc, _score in retrieved]
    filters = retrieval["filters"]
    ambiguous = retrieval["ambiguous_assignee_candidates"]
    sources = sorted({doc.metadata["issue_key"] for doc in docs})

    if not docs:
        answer = (
            "I could not find enough information in the PowerOps knowledge base "
            "to answer this question reliably. This request should be escalated "
            "to DevOps management."
        )
    else:
        context = build_context(docs)
        messages = [
            ("system", SYSTEM_PROMPT),
            ("human", f"PowerOps Context:\n\n{context}\n\nQuestion: {question}"),
        ]
        answer = llm.invoke(messages).content

    escalate, reasons, evidence = should_escalate(question, retrieved, filters, ambiguous, answer)

    escalation_record = None
    if escalate:
        escalation_record = create_escalation(question, reasons, evidence, retrieved_docs=docs, user=user)

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "filters": filters,
        "retrieved_count": len(docs),
        "needs_escalation": escalate,
        "escalation": escalation_record,
    }


print("ask_powerops() (v2, with escalation) defined.")


ask_powerops() (v2, with escalation) defined.


## 7. Test: answerable questions (should NOT escalate)


In [7]:
ANSWERABLE_QUESTIONS = [
    "Which high-priority problems belong to Nova Team?",
    "What is happening with INO-21920?",
]

def print_response(resp: dict) -> None:
    print("=" * 78)
    print(f"Q: {resp['question']}")
    print("-" * 78)
    print(resp["answer"])
    print("-" * 78)
    print(f"Sources: {resp['sources']}")
    print(f"Filters: {resp['filters']}")
    print(f"Retrieved count: {resp['retrieved_count']} | needs_escalation: {resp['needs_escalation']}")
    if resp["escalation"]:
        print(f"Escalation: {resp['escalation']['escalation_id']} — {resp['escalation']['reason']}")
    print()


for q in ANSWERABLE_QUESTIONS:
    print_response(ask_powerops(q))


Q: Which high-priority problems belong to Nova Team?
------------------------------------------------------------------------------
Here are the high-priority problems that belong to the Nova Team:

| Issue Key  | Summary                                                   | Team       | Assignee            | Priority | Status |
|------------|-----------------------------------------------------------|------------|---------------------|----------|--------|
| INO-20547  | PCP5 - System Logs are not logging the issues from EB     | Nova Team  | Myers, Deepa        | High     | Done   |
| INO-19996  | PCP8: Increase the JVM of EB batch chunker GenericBatch   | Nova Team  | Myers, Deepa        | High     | Done   |
| INO-19870  | Donna Trivedi Linux User Access Review Due 04-20-2026    | Nova Team  | Graham, Aditya      | High     | Done   |
| INO-19871  | Donna Rice Linux User Access Review Due 04-20-2026       | Nova Team  | Chawla, Gregory     | High     | Done   |
| INO-19775  | April 20

Q: What is happening with INO-21920?
------------------------------------------------------------------------------
The issue INO-21920 is related to SDX cases that are being created but are remaining in a "Delayed Processing Pending" status. It is currently assigned to the Falcon Squad and is being worked on by Deepa Sullivan (Contractor). The priority of this issue is Medium, and its status is In Progress.
------------------------------------------------------------------------------
Sources: ['INO-21920']
Filters: {'issue_key': 'INO-21920'}
Retrieved count: 1 | needs_escalation: False



## 8. Test: questions that should escalate

These are the same questions from Notebook 05 where `needs_escalation` was
incorrectly `False`. Watch it flip to `True` here, with a concrete reason
and a persisted escalation record.


In [8]:
ESCALATION_QUESTIONS = [
    "What critical issues are currently open?",           # no literal 'critical'/'open' -> LLM should decline
    "What issues does John currently own?",                # person not in dataset
    "What is happening with OPS-1015?",                    # wrong issue-key prefix, not in dataset
    "Which high-priority problems belong to Team Beta?",   # team doesn't exist
    "What issues does Anjali have?",                       # ambiguous assignee (3 people)
]

for q in ESCALATION_QUESTIONS:
    print_response(ask_powerops(q))


Q: What critical issues are currently open?
------------------------------------------------------------------------------
Sufficient information was not found in the PowerOps knowledge base to identify any currently open critical issues. All listed issues are marked as "Done."
------------------------------------------------------------------------------
Sources: ['INO-19775', 'INO-19861', 'INO-19863', 'INO-19866', 'INO-19871']
Filters: {}
Retrieved count: 5 | needs_escalation: True
Escalation: ESC-0001 — The language model indicated the retrieved context was insufficient to answer confidently.



Q: What issues does John currently own?
------------------------------------------------------------------------------
Sufficient information was not found in the PowerOps knowledge base to determine which issues John currently owns.
------------------------------------------------------------------------------
Sources: ['INO-20754', 'INO-20755', 'INO-20769', 'INO-21728', 'INO-21731']
Filters: {}
Retrieved count: 5 | needs_escalation: True
Escalation: ESC-0002 — The language model indicated the retrieved context was insufficient to answer confidently.



Q: What is happening with OPS-1015?
------------------------------------------------------------------------------
Sufficient information was not found in the PowerOps knowledge base.
------------------------------------------------------------------------------
Sources: ['INO-19991', 'INO-20247', 'INO-20447', 'INO-21038', 'INO-21869']
Filters: {}
Retrieved count: 5 | needs_escalation: True
Escalation: ESC-0003 — The language model indicated the retrieved context was insufficient to answer confidently.



Q: Which high-priority problems belong to Team Beta?
------------------------------------------------------------------------------
Sufficient information was not found in the PowerOps knowledge base regarding Team Beta.
------------------------------------------------------------------------------
Sources: ['INO-20130', 'INO-20216', 'INO-20547', 'INO-21071', 'INO-21149']
Filters: {'priority': 'High'}
Retrieved count: 5 | needs_escalation: True
Escalation: ESC-0004 — The language model indicated the retrieved context was insufficient to answer confidently.



Q: What issues does Anjali have?
------------------------------------------------------------------------------
Here are the issues related to Anjali Porter:

| Issue Key   | Summary                                      | Team         | Assignee                | Priority | Status      |
|-------------|----------------------------------------------|--------------|-------------------------|----------|-------------|
| INO-21689   | Access to Jennifer Simpson Console for Anjali Porter (ncid: jpatel1) | Nova Team    | Porter, Anjali (Contractor) | Medium   | Soft Delete |
| INO-21639   | Access to Bitbucket repos for Anjali Porter (ncid: jpatel1) | Summit Crew  | Rice, Anjali (Contractor)   | Medium   | Done       |
| INO-21636   | Access to Linux servers for Anjali Porter (ncid: jpatel1) | Summit Crew  | Rice, Anjali (Contractor)   | Medium   | Done       |
| INO-21637   | Access to database for Anjali Porter (ncid: jpatel1) | Summit Crew  | Tucker, Eric              | Medium   | Done     

## 9. Inspect the persisted escalation log

In [9]:
escalations = _load_escalations()
print(f"Total escalations on file: {len(escalations)}")
print()
for esc in escalations[-5:]:
    print(json.dumps(esc, indent=2))
    print("-" * 60)


Total escalations on file: 5

{
  "escalation_id": "ESC-0001",
  "timestamp": "2026-08-23T17:51:14.578576+00:00",
  "question": "What critical issues are currently open?",
  "user": null,
  "reason": "The language model indicated the retrieved context was insufficient to answer confidently.",
  "reasons": [
    "The language model indicated the retrieved context was insufficient to answer confidently."
  ],
  "retrieved_issue_ids": [
    "INO-19775",
    "INO-19861",
    "INO-19863",
    "INO-19866",
    "INO-19871"
  ],
  "evidence": {
    "num_retrieved": 5,
    "zero_results": false,
    "max_similarity": 0.429070562,
    "requested_issue_key_not_found": false,
    "requested_filters_not_reflected": false,
    "ambiguous_assignee": false,
    "llm_declined": true,
    "unsupported_citations": []
  },
  "status": "Pending Management Review"
}
------------------------------------------------------------
{
  "escalation_id": "ESC-0002",
  "timestamp": "2026-08-23T17:51:15.510538+00:00"

## Summary & next steps

- Built `evaluate_evidence()` — six deterministic, checkable signals (zero
  results, requested issue key missing, requested filter combo not
  reflected, ambiguous assignee, LLM-declined phrase match, unsupported
  citations) instead of relying on the LLM's self-reported confidence.
- Deliberately did **not** use raw cosine-similarity thresholding as an
  escalation trigger — this dataset's short Summaries compress the
  similarity range enough that a fixed cutoff would be unreliable; it's
  reported (`max_similarity`) for visibility only.
- Built `should_escalate()` and `create_escalation()`, persisting to
  `data/escalations.json` with timestamp, question, reasons, retrieved
  issue IDs, full evidence dict, and `Pending Management Review` status.
- Rebuilt `ask_powerops()` to check every answer — not just zero-retrieval
  cases — against `should_escalate()`. Re-running Notebook 05's problem
  questions here shows `needs_escalation` correctly flip to `True`,
  closing the exact gap that notebook surfaced.

**Next: Notebook 09 (Streamlit UI)** wires `ask_powerops()` into an
interactive app — question input, grounded answer, source issues, applied
filters, and an escalation-required panel when `needs_escalation` is `True`.
(Prompt 10's `src/` refactor will then extract everything built across
Notebooks 01–06 into reusable modules so the app doesn't duplicate this
logic.)
